In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS electricity.landing.checkpoints")

In [0]:
base = "/Volumes/electricity/landing/fingrid_raw"
chk = "/Volumes/electricity/landing/checkpoints"

for ds in ["production", "consumption", "wind"]:
    (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("sep", ";")
        .option("cloudFiles.schemaLocation", f"{chk}/{ds}/schema")
        .load(f"{base}/{ds}/")
        .selectExpr("*",
                    "_metadata.file_path AS source_file",
                    "current_timestamp() AS ingested_at")
        .writeStream
        .option("checkpointLocation", f"{chk}/{ds}/checkpoint")
        .trigger(availableNow=True)
        .toTable(f"electricity.bronze.{ds}")
        .awaitTermination())

In [0]:
for f in dbutils.fs.ls("/Volumes/electricity/landing/fingrid_raw/"):
    print(f.name)
    for g in dbutils.fs.ls(f.path):
        print("   ", g.name)

In [0]:
import os
base = "/Volumes/electricity/landing/fingrid_raw"
os.makedirs(f"{base}/wind", exist_ok=True)
for f in os.listdir(f"{base}/Wind generation"):
    os.rename(f"{base}/Wind generation/{f}", f"{base}/wind/{f}")
os.rmdir(f"{base}/Wind generation")

In [0]:
import os
base = "/Volumes/electricity/landing/fingrid_raw"

if os.path.exists(f"{base}/Consumption"):
    os.rename(f"{base}/Consumption", f"{base}/consumption")

In [0]:
%sql
SELECT 'production' AS ds, COUNT(*) FROM electricity.bronze.production
UNION ALL SELECT 'consumption', COUNT(*) FROM electricity.bronze.consumption
UNION ALL SELECT 'wind', COUNT(*) FROM electricity.bronze.wind;

In [0]:
%sql
SELECT source_file,
       datasetId,
       COUNT(*)                  AS row_count,
       COUNT(DISTINCT startTime) AS distinct_times,
       MIN(startTime)            AS first_time,
       MAX(startTime)            AS last_time
FROM electricity.bronze.consumption
GROUP BY source_file, datasetId;

In [0]:
%sql
DELETE FROM electricity.bronze.consumption
WHERE source_file LIKE '%/Consumption/%';

In [0]:
%sql
SELECT COUNT(*) FROM electricity.bronze.consumption;  

In [0]:
%sql
DESCRIBE HISTORY electricity.bronze.consumption